# F4-multivar-calculus — Session 2: The Chain Rule, tanh, and Sums of Squares

*One class session, roughly 90 minutes. Builds on Session 1 (partials,
gradients, central differences).*

**This session:** the multivariable chain rule built directly from the
Calc AB one — as bookkeeping of nudges, where *every route from the input
to the output contributes* — the tanh function fully worked from its
exponential definition (the exam's favorite derivative), and the gradient
of a component-form sum of squares, the single most exam-relevant
computation in this unit.
Two fully worked exam-style examples run the real registers: a normal-form
multiple choice and a banned-API coding task.

Try every checkpoint by hand first, then verify with NumPy.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. The 1-D Chain Rule, Retold as Nudge Accounting

**Motivation.**
You know the chain rule from Calc AB:
$\big(f(g(t))\big)' = f'(g(t))\cdot g'(t)$.
To extend it to several variables we first *retell* it in a form that
extends by itself: as bookkeeping of tiny nudges.

**The nudge story.**
Nudge $t$ by a tiny $dt$.

1. The inner function moves: $g$ changes by about $g'(t)\,dt$.
2. That change is itself a nudge to the input of $f$: $f$ changes by about
   $f'(g(t)) \cdot \big(g'(t)\,dt\big)$.

Divide by $dt$: the rate of the composite is $f'(g(t))\, g'(t)$.
**Along a route, rates multiply** — each stage scales the nudge passing
through it.
That is all the chain rule says.

**Worked example.**
$\dfrac{d}{dt}\,(t^2 + 1)^5$: the outer rate is $5(\cdot)^4$ at the inner
value, the inner rate is $2t$, so the derivative is
$5(t^2 + 1)^4 \cdot 2t = 10t\,(t^2 + 1)^4$.
At $t = 1.3$ the code confirms it against a central difference
(Session 1's checker):

In [ ]:
def composite_1d(t):
    return (t**2 + 1)**5


t0, h = 1.3, 1e-6
hand = 10 * t0 * (t0**2 + 1)**4
numeric = (composite_1d(t0 + h) - composite_1d(t0 - h)) / (2 * h)
print("hand   :", hand)
print("numeric:", numeric)
print("gap    :", abs(hand - numeric))

### Checkpoint 1

1. By hand: $\dfrac{d}{dt}\,(t^3 + 2t)^4$.
2. By hand: $\dfrac{d}{dt}\, e^{-t^2}$.
3. Nudge story: along some route, the inner stage scales nudges by $3$
   and the outer stage scales them by $-2$.
   What is the composite's rate, and which single word of the story gives
   the answer?

## 2. The Multivariable Chain Rule: Every Route Contributes

**Motivation.**
Now let the outside world feed *several* inputs:
$f(u, v)$ where both $u = u(t)$ and $v = v(t)$ move as $t$ moves.
How fast does $f(u(t), v(t))$ change?
There are now **two routes** from $t$ to the output:
$$t \to u \to f \qquad\text{and}\qquad t \to v \to f .$$

**The nudge accounting.**
Nudge $t$ by $dt$.
*Both* inputs move: $u$ by $u'(t)\,dt$ and $v$ by $v'(t)\,dt$.
Session 1's partials price each input's nudge — $\partial f/\partial u$
per unit of $u$-nudge, $\partial f/\partial v$ per unit of $v$-nudge —
and the two contributions **add**:
$$df \approx \frac{\partial f}{\partial u}\, u'(t)\,dt
   + \frac{\partial f}{\partial v}\, v'(t)\,dt .$$

**Definition (multivariable chain rule).**
$$\frac{d}{dt}\, f\big(u(t), v(t)\big)
  = \frac{\partial f}{\partial u}\,\frac{du}{dt}
  + \frac{\partial f}{\partial v}\,\frac{dv}{dt},$$
and in general, for $f$ of $d$ inputs fed by a path
$r(t) = (r_0(t), \dots, r_{d-1}(t))$,
$$\frac{d}{dt}\, f\big(r(t)\big)
  = \sum_{k=0}^{d-1} \frac{\partial f}{\partial p_k}\bigg|_{r(t)}
    \cdot r_k'(t)
  \;=\; \nabla f \cdot r'(t).$$
One term per route — *rates multiply along each route, and routes add.*

**Why every route must appear.**
Each term is a real physical contribution: when $t$ nudges, $u$ really
moves and $v$ really moves, and $f$ feels both.
Dropping a term means pretending one input held still when it did not.
(The only honest way a term vanishes is when its own factors kill it:
a frozen route, $r_k'(t) = 0$, or an indifferent output,
$\partial f/\partial p_k = 0$.)

**Worked example, both ways.**
$f(u, v) = u^2 v$ with $u = 3t$, $v = t^2$.

*Via the chain rule:*
$f_u = 2uv$ and $f_v = u^2$; $\;u' = 3$ and $v' = 2t$.
$$\frac{d}{dt} f = (2uv)(3) + (u^2)(2t)
  = 2(3t)(t^2)\cdot 3 + (3t)^2\, 2t = 18t^3 + 18t^3 = 36t^3 .$$
Note the two routes contribute *equally* here — drop either one and you
get $18t^3$, off by a factor of 2.

*Via substitute-first:* $f(3t, t^2) = (3t)^2 t^2 = 9t^4$, whose ordinary
derivative is $36t^3$.
The two computations must always agree — substitute-first is the
ground truth the chain rule reproduces without needing a closed form.

In [ ]:
def f2(u, v):
    return u**2 * v


def composite(t):
    return f2(3 * t, t**2)


t0 = 1.2
u0, v0 = 3 * t0, t0**2
via_chain = (2 * u0 * v0) * 3 + (u0**2) * (2 * t0)
via_subst = 36 * t0**3
numeric = (composite(t0 + 1e-6) - composite(t0 - 1e-6)) / 2e-6

print("via chain rule :", via_chain)
print("via substitute :", via_subst)
print("central diff   :", numeric)
print("one route only :", 18 * t0**3, "  <- off by a factor of 2")

### Checkpoint 2

1. $f(u, v) = u + v^2$ with $u = t^3$, $v = 2t$: compute
   $\frac{d}{dt} f(u(t), v(t))$ twice — via the chain rule and via
   substitute-first — and check they agree.
2. One sentence: why does a route with $r_k'(t) = 0$ contribute nothing,
   in nudge language?
3. $f$ has three inputs, each depending on $t$.
   How many terms does $\frac{d}{dt} f(r(t))$ have, and what does each
   term's pair of factors represent?

## 3. tanh, Fully Worked

**Motivation.**
The exam's calculus cluster has one star function: the **hyperbolic
tangent**.
It squashes any real number into $(-1, 1)$ smoothly, and its derivative
has a closed form so clean that deriving it is a classic
reasoning-required exam item.
We derive everything from scratch.

**Definition.**
$$\tanh x = \frac{e^x - e^{-x}}{e^x + e^{-x}} .$$
First facts, straight from the formula:

- $\tanh 0 = 0$ (the numerator vanishes);
- as $x \to +\infty$: $e^{-x} \to 0$, so
  $\tanh x \to e^x / e^x = 1$ (from below);
- as $x \to -\infty$: symmetric story, $\tanh x \to -1$;
- odd symmetry: $\tanh(-x) = -\tanh x$ (swap the exponentials).

**Deriving the derivative (the full quotient-rule route).**
Name the numerator and denominator:
$$N(x) = e^x - e^{-x}, \qquad D(x) = e^x + e^{-x}.$$
Differentiating term by term (chain rule on $e^{-x}$: derivative
$-e^{-x}$):
$$N'(x) = e^x + e^{-x} = D(x), \qquad D'(x) = e^x - e^{-x} = N(x).$$
Each is the other's derivative — that is the engine of the whole
computation.
Now the quotient rule:
$$\tanh'(x) = \frac{N'D - N D'}{D^2} = \frac{D^2 - N^2}{D^2}
  = 1 - \frac{N^2}{D^2} = 1 - \tanh^2(x).$$
For the alternative closed form, expand the squares:
$$D^2 = e^{2x} + 2 + e^{-2x}, \qquad N^2 = e^{2x} - 2 + e^{-2x}
  \;\Longrightarrow\; D^2 - N^2 = 4,$$
so equivalently
$$\tanh'(x) = \frac{4}{\left(e^x + e^{-x}\right)^2}.$$

**The identity's consequences (reasoning-required register).**
*Claim: $0 < \tanh'(x) \le 1$ for every real $x$, with equality only at
$x = 0$.*

- *Strictly positive.* Both $e^x > 0$ and $e^{-x} > 0$, so
  $|N| = |e^x - e^{-x}| < e^x + e^{-x} = D$.
  Hence $\tanh^2 x = N^2/D^2 < 1$, and
  $\tanh'(x) = 1 - \tanh^2(x) > 0$: tanh is strictly increasing
  everywhere.
- *At most 1.* $\tanh^2 x \ge 0$ always, so $\tanh' \le 1$; equality
  needs $\tanh x = 0$, i.e. $e^x = e^{-x}$, which forces $x = 0$.
  The derivative's maximum value is exactly $1$, attained only at the
  origin.
- *Saturation.* As $x \to \pm\infty$, $\tanh x \to \pm 1$, so
  $\tanh' = 1 - \tanh^2 \to 0$: far from the origin the function barely
  moves — it is *saturated*.

Every claim above is checkable by machine; the code plots both curves and
verifies the identity against central differences at several points.

In [ ]:
x = np.linspace(-4, 4, 400)

plt.figure(figsize=(6.5, 3.8))
plt.plot(x, np.tanh(x), label=r"$\tanh x$")
plt.plot(x, 1 - np.tanh(x)**2, label=r"$\tanh'(x) = 1 - \tanh^2 x$")
plt.axhline(1, color="gray", linewidth=0.5, linestyle=":")
plt.axhline(-1, color="gray", linewidth=0.5, linestyle=":")
plt.axhline(0, color="gray", linewidth=0.5)
plt.xlabel("x")
plt.title("tanh squashes into (-1, 1); its slope peaks at 1 and saturates to 0")
plt.legend()
plt.show()

In [ ]:
h = 1e-6
pts = np.array([-3.0, -1.0, 0.0, 0.7, 2.5])
direct = (np.tanh(pts + h) - np.tanh(pts - h)) / (2 * h)   # slope, measured
identity = 1 - np.tanh(pts)**2                             # slope, claimed
closed = 4 / (np.exp(pts) + np.exp(-pts))**2               # the 4/D^2 form

print("measured :", direct)
print("identity :", identity)
print("4/D^2    :", closed)
print("max gap  :", np.abs(direct - identity).max())

### Checkpoint 3

1. If $\tanh(x_0) = 0.6$, compute $\tanh'(x_0)$ without finding $x_0$.
2. One line: why can $\tanh'(x)$ never be negative?
3. Where is $\tanh'$ largest, what is its value there, and what happens
   to $\tanh'$ as $|x|$ grows?

## 4. Chaining Through tanh: Squashed Weighted Sums

**Motivation.**
On the exam, tanh rarely stands alone: it squashes a *weighted sum*,
$$g(w) = \tanh\Big(\sum_{k=0}^{d-1} a_k w_k\Big),$$
with constants $a_k$ and adjustable coefficients $w_k$ — and the question
is the gradient with respect to $w$ (that is p09's shape, and p14 and p16
build on it).

**The two-stage chain, per coordinate.**
Name the inner scalar $s = \sum_k a_k w_k$.
The route from $w_j$ to the output passes through $s$:

1. inner stage: $\dfrac{\partial s}{\partial w_j} = a_j$
   (scan the sum — only the $k = j$ term contains $w_j$);
2. outer stage: $\dfrac{d}{ds}\tanh(s) = 1 - \tanh^2(s)$.

Rates multiply along the route:
$$\frac{\partial g}{\partial w_j} = \big(1 - \tanh^2(s)\big)\, a_j
\qquad\Longrightarrow\qquad
\nabla g = \big(1 - \tanh^2(s)\big)\, a .$$
The whole gradient is the constant vector $a$ rescaled by *one shared
number* — the squashing stage's slope at the current $s$.

**Worked example.**
$a = (2, -1)$, $w = (0.5, 1)$: then $s = 1 - 1 = 0$, the slope factor is
$1 - \tanh^2(0) = 1$, and $\nabla g = (2, -1)$ — at $s = 0$ the squashing
is momentarily invisible.
Push $w$ to $(2, 1)$ instead: $s = 3$, the factor drops to
$1 - \tanh^2(3) \approx 0.0099$, and the gradient all but vanishes —
**saturation kills gradients**, the fact behind p16's "alarmed engineers"
scenario.

In [ ]:
def num_gradient(f, point, h=1e-6):
    """Session 1's checker, restated: nudge ONE coordinate at a time."""
    point = np.asarray(point, dtype=float)
    grad_est = np.zeros_like(point)
    for j in range(point.shape[0]):
        step = np.zeros_like(point)
        step[j] = h
        grad_est[j] = (f(point + step) - f(point - step)) / (2 * h)
    return grad_est


a = np.array([2.0, -1.0])


def g(w):
    return np.tanh(np.sum(a * w))


for w in (np.array([0.5, 1.0]), np.array([2.0, 1.0])):
    s = np.sum(a * w)
    grad_hand = (1 - np.tanh(s)**2) * a
    grad_num = num_gradient(g, w)
    print(f"s = {s:4.1f}   slope factor = {1 - np.tanh(s)**2:.4f}")
    print("  hand   :", grad_hand)
    print("  numeric:", grad_num, "  max gap:",
          np.abs(grad_hand - grad_num).max())

### Checkpoint 4

1. Hand-derive $\partial g/\partial w_1$ for
   $g(w) = \tanh(4w_0 - w_1 + 2w_2)$.
2. The inner sum currently sits at $s = 5$ (so $\tanh s \approx 0.9999$).
   Roughly how large is the slope factor $1 - \tanh^2(s)$, and what does
   that mean — in one sentence — for how much any $w_j$ can move the
   output?

## 5. Sums of Squares, Component Form

**Motivation.**
Here is the exam's canonical gradient target.
You have a data grid `X` of shape $(N, d)$ — row $n$ holds the numbers
$X_{n,0}, \dots, X_{n,d-1}$ — targets $y_n$, and adjustable coefficients
$w_0, \dots, w_{d-1}$.
The prediction rule for row $n$ is the weighted sum
$\sum_k X_{n,k} w_k$, and the **mismatch score** is the averaged sum of
squared misses:
$$Q(w) = \frac{1}{N}\sum_{n=0}^{N-1}
  \Big(y_n - \sum_{k=0}^{d-1} X_{n,k} w_k\Big)^{2}.$$
Small $Q$ means the rule's outputs sit close to the targets.
The gradient of $Q$ tells us how each coefficient moves the score — and
deriving it is a pure chain-rule exercise in component form.

**Warm-up: one coefficient.**
$Q(w) = \sum_n (y_n - w x_n)^2$ (no $1/N$, single $w$).
Chain per term: outer $(\cdot)^2$ gives $2(y_n - w x_n)$; inner
$\frac{d}{dw}(y_n - w x_n) = -x_n$.
$$Q'(w) = \sum_n 2\,(y_n - w x_n)\cdot(-x_n)
        = -2 \sum_n (y_n - w x_n)\, x_n .$$
**The minus sign comes from the inner derivative** — the target $y_n$ is
constant, and $w$ enters with a minus in front of its route.

**The full component-form derivation.**
Name the residuals $r_n = y_n - \sum_k X_{n,k} w_k$ (the misses), so
$Q = \frac1N \sum_n r_n^2$.
Differentiate with respect to one chosen $w_j$, route by route:

1. outer stage, term $n$: $\dfrac{\partial}{\partial r_n} r_n^2 = 2r_n$;
2. inner stage: $\dfrac{\partial r_n}{\partial w_j} = -X_{n,j}$ — scan
   the sum $\sum_k X_{n,k} w_k$; only the $k = j$ term contains $w_j$,
   and it carries the leading minus;
3. every row $n$ is a separate route from $w_j$ to $Q$, so the $N$
   contributions add.

$$\boxed{\;\frac{\partial Q}{\partial w_j}
  = -\frac{2}{N} \sum_{n=0}^{N-1} r_n\, X_{n,j}\;}
\qquad j = 0, \dots, d-1 .$$
Read it aloud: *residual times the $j$-th column's entries, summed over
the rows, scaled by $-2/N$.*
Note which index does what: $n$ is summed away; $j$ stays free — one
partial per coefficient.

**Two-coefficient special case** (p13's shape): predictions
$w_0 + w_1 h_n$ give
$$\frac{\partial Q}{\partial w_0} = -\frac{2}{N}\sum_n r_n,
\qquad
\frac{\partial Q}{\partial w_1} = -\frac{2}{N}\sum_n r_n h_n$$
— the constant coefficient's "column" is all ones.

The code runs the warm-up on tiny data where every number is checkable by
eye.

In [ ]:
x_data = np.array([1.0, 2.0, 3.0])
y_data = np.array([2.0, 3.0, 5.0])


def Q1(w):
    return np.sum((y_data - w * x_data)**2)


w0 = 1.0
r = y_data - w0 * x_data          # residuals: (1, 1, 2)
hand = -2 * np.sum(r * x_data)    # -2 (1*1 + 1*2 + 2*3) = -18
numeric = (Q1(w0 + 1e-6) - Q1(w0 - 1e-6)) / 2e-6

print("residuals:", r)
print("hand Q'(1):", hand)
print("numeric   :", numeric)

### Checkpoint 5

1. By hand, warm-up form: $x = (2, 1)$, $y = (3, 4)$,
   $Q(w) = \sum_n (y_n - w x_n)^2$.
   Compute $Q'(w)$ at $w = 1$.
2. Derive $\partial Q/\partial w_0$ for the two-coefficient score
   $Q(w_0, w_1) = \frac{1}{N}\sum_n (s_n - w_0 - w_1 h_n)^2$, showing
   where its inner derivative $-1$ comes from.
3. In $\partial Q/\partial w_j = -\frac{2}{N}\sum_n r_n X_{n,j}$: which
   index is summed away, which stays free, and why does that match the
   gradient having $d$ entries?

## 6. Worked Exam-Style Example I: Normal-Form Multiple Choice

Exam MC items about these gradients wrap the number in a **normal form**
so exactly one decoded answer is right.
Here is one in full, in the real register.

---

**Problem (reasoning is not required; no code needed).**
For data points $x = (1, 3)$, $y = (2, 4)$ define
$$Q(w) = \sum_{n=1}^{2} (y_n - w\,x_n)^2 .$$
Compute $Q'(w)$ at $w = 1$.
Your answer is a nonzero integer that can be written as $s \cdot m$ with
$s \in \{+1, -1\}$ and $m$ a positive integer.
What is $s + m$?

A. 3  B. 5  C. 7  D. 9  E. 11

---

**Solution.**

*Step 1 — write the derivative in the warm-up form.*
$Q'(w) = -2\sum_n (y_n - w x_n)\, x_n$.

*Step 2 — evaluate the residuals at $w = 1$.*
$r_1 = 2 - 1 = 1$, $\;r_2 = 4 - 3 = 1$.

*Step 3 — assemble.*
$Q'(1) = -2\,(1 \cdot 1 + 1 \cdot 3) = -8$.

*Step 4 — decode the normal form.*
$-8 = s \cdot m$ with $s = -1$, $m = 8$, so $s + m = -1 + 8 = 7$:
**answer C**.
The decomposition is unique for a nonzero integer — that is what makes
the encoding gradable; sign errors do not just flip your number, they
land on a *different* decoded value ($+8$ gives $1 + 8 = 9$, choice D —
the trap is built in).

*Step 5 — the free numeric cross-check* (allowed on the real exam
whenever code is allowed at all): a central difference of $Q$ at $w = 1$
must land on $-8$.

In [ ]:
xd = np.array([1.0, 3.0])
yd = np.array([2.0, 4.0])


def Qm(w):
    return np.sum((yd - w * xd)**2)


hand = -2 * np.sum((yd - 1.0 * xd) * xd)
numeric = (Qm(1.0 + 1e-6) - Qm(1.0 - 1e-6)) / 2e-6
s_dec, m_dec = -1, 8

print("hand Q'(1):", hand, "   numeric:", numeric)
print("decode: s =", s_dec, " m =", m_dec, " ->  s + m =", s_dec + m_dec)

### Checkpoint 6

1. Same register, new numbers: $x = (2, 1)$, $y = (1, 3)$, evaluate
   $Q'(w)$ at $w = 2$ and decode $s + m$.
2. Why does the $s \cdot m$ encoding force everyone who computed
   correctly to submit the *same* option — and why must the problem
   guarantee the answer is nonzero for the encoding to work?

## 7. Worked Exam-Style Example II: Constrained Coding

The other exam register: implement the component-form gradient under an
API ban, then prove it right with a numeric checker.
Solved step by step below.

---

**Problem.**
Given a data grid `X` of shape `(40, 2)`, targets `y` of shape `(40,)`,
and coefficients `w` of shape `(2,)` (seeded data below), with
$$Q(w) = \frac{1}{N}\sum_n \Big(y_n - \sum_k X_{n,k} w_k\Big)^2,$$
write `q_gradient(X, y, w)` returning the shape-`(2,)` array of partials
$\partial Q/\partial w_j$.
**Banned (zero points): `@`, `np.matmul`, `np.dot`, `.T`, any loop.**
Use elementwise multiplication, broadcasting, and axis sums only.
Then compute `check_gap`, the max abs difference against central
differences (the checker MAY loop).

---

**Solution.**

*Step 1 — restate the target formula (Section 5):*
$\partial Q/\partial w_j = -\frac{2}{N}\sum_n r_n X_{n,j}$ with
$r_n = y_n - \sum_k X_{n,k} w_k$.

*Step 2 — predictions without a banned product.*
`X * w` broadcasts `(40, 2) * (2,)` to `(40, 2)` — each row multiplied
elementwise by `w` — and `.sum(axis=1)` collapses each row's $k$-sum:
shape `(40,)`.

*Step 3 — residuals.* `r = y - pred`, shape `(40,)`.

*Step 4 — all partials at once.*
$\sum_n r_n X_{n,j}$ for every $j$ simultaneously: reshape `r` to
`(40, 1)`, multiply into `X` (broadcast to `(40, 2)` — row $n$ scaled by
$r_n$), then sum down the rows with `.sum(axis=0)`: shape `(2,)`.
Scale by `-2/N`.

*Step 5 — self-grade against the contract.*
Output shape `(2,)` ✓; no `@`, no `np.matmul`, no `np.dot`, no `.T`, no
loop in `q_gradient` ✓; the loop lives only in the checker, which the
statement explicitly allows.

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
X = rng.normal(0, 1, (40, 2))
y = rng.normal(0, 1, 40)
w = np.array([1.0, -0.5])


def q_gradient(X, y, w):
    pred = (X * w).sum(axis=1)          # (40, 2) * (2,) -> row sums, (40,)
    r = y - pred                        # residuals, (40,)
    return -(2 / X.shape[0]) * (r[:, None] * X).sum(axis=0)   # (2,)


grad = q_gradient(X, y, w)
print("gradient:", grad, "  shape:", grad.shape)

In [ ]:
def Q_full(wv):
    return np.mean((y - (X * wv).sum(axis=1))**2)


h = 1e-6
check_gap = 0.0
for j in range(w.shape[0]):             # the checker MAY loop
    step = np.zeros_like(w)
    step[j] = h
    est = (Q_full(w + step) - Q_full(w - step)) / (2 * h)
    check_gap = max(check_gap, abs(est - grad[j]))

print("check_gap:", check_gap)          # ~1e-9: formula and code agree

### Checkpoint 7

1. The tempting one-liner for $\sum_n r_n X_{n,j}$ is `np.dot(r, X)`.
   Which part of the ban does it hit, and what is the legal broadcasting
   rewrite?
2. Predict the shapes of `r[:, None]`, of `r[:, None] * X`, and of the
   final `.sum(axis=0)` result.
3. Why is the loop acceptable in the checker but worth zero points in
   `q_gradient` itself?

## 8. Common Pitfalls II

**Pitfall 1 — dropping the minus sign from the inner derivative.**
The residual is $y_n - \sum_k X_{n,k} w_k$: the coefficient's route
enters *behind a minus*, so the inner derivative is $-X_{n,j}$.
Forget it and your gradient points exactly the wrong way — downhill
becomes uphill.
The checker catches it as a gap of *twice* the gradient's size:

In [ ]:
r_ok = y - (X * w).sum(axis=1)

grad_broken = (2 / X.shape[0]) * (r_ok[:, None] * X).sum(axis=0)   # BROKEN: no minus
grad_fixed = -(2 / X.shape[0]) * (r_ok[:, None] * X).sum(axis=0)

num = np.array([(Q_full(w + np.array([1e-6, 0])) - Q_full(w - np.array([1e-6, 0]))) / 2e-6,
                (Q_full(w + np.array([0, 1e-6])) - Q_full(w - np.array([0, 1e-6]))) / 2e-6])

print("broken:", grad_broken, "  gap:", np.abs(grad_broken - num).max())
print("fixed :", grad_fixed, "  gap:", np.abs(grad_fixed - num).max())

A wrong-signed gradient is worse than a random one: every "downhill
nudge" taken with it *increases* the score.
Sanity probe: nudge one coefficient a tiny step against the sign of your
partial and confirm $Q$ actually falls.

**Pitfall 2 — dropping the $2$ (or the whole $2/N$).**
The outer stage of the chain contributes the $2$; the average
contributes $1/N$.
A gradient missing either factor is *proportional* to the truth —
same direction, wrong length — and hand reasoning will never notice.
The numeric checker will:

In [ ]:
grad_noscale = -(r_ok[:, None] * X).sum(axis=0)      # BROKEN: dropped 2/N

print("broken :", grad_noscale)
print("fixed  :", grad_fixed)
print("ratio  :", grad_noscale / grad_fixed)          # N/2 = 20, both entries
print("gap    :", np.abs(grad_noscale - num).max())   # large -> caught

"Proportional" is graded as *wrong*: exam checkers compare numbers, not
directions.
Keep the bookkeeping visible — write the $-2/N$ into the formula before
touching the keyboard.

**Pitfall 3 — summing over the wrong index.**
$\sum_n r_n X_{n,j}$ sums over rows ($n$), leaving one number per
*column* ($j$).
Summing over `axis=1` instead collapses the columns and leaves shape
`(N,)` — the shape contract catches it before any numbers get compared:

In [ ]:
grad_wrong_axis = -(2 / X.shape[0]) * (r_ok[:, None] * X).sum(axis=1)   # BROKEN

print("broken shape:", grad_wrong_axis.shape, "  (contract says (2,))")
print("fixed  shape:", grad_fixed.shape)

Predict the shape *before* running: $d$ coefficients need $d$
partials — if a 40-entry array comes back from 2 coefficients, the axis
is wrong no matter how plausible the numbers look.
(Session 1's F2 habit, unchanged.)

### Checkpoint 8

1. A classmate's `q_gradient` passes the shape contract, but every entry
   is exactly $-1$ times the checker's estimate.
   Which pitfall is this, and which single character fixes it?
2. Another passes the shape contract and the direction looks right, but
   each entry is exactly $20\times$ the checker's estimate (with
   $N = 40$).
   Which factor was dropped?
3. Without running: `r[:, None] * X` has shape `(40, 2)`.
   Which axis must be summed to honor the contract, and what does the
   *other* axis's sum compute instead?

## Exam Connections

How this unit's material shows up in Round 1 (paraphrased from the
`reference/analysis.md` topic table — no real test text here):

- The **calculus cluster** is small but reliably present: r1-2026 carried
  one 5-point reasoning-required sub-part on exactly Section 3's
  material — the tanh derivative via the exponential definition and the
  $1 - \tanh^2$ identity.
  The analysis's difficulty profile lists it among the points reachable
  straight from the Calc AB baseline plus this unit; p03, p11, and p16
  train the register.
- The **NumPy implementation cluster** (8 sub-parts, 55 points in
  r1-2026) includes broadcasting-only gradient implementations with
  explicit API bans and zero-point clauses — Section 7's exact shape.
  p07, p09, p10, p14, and p17 drill it.
- The analysis also notes that *single-variable calculus is assumed but
  chain-rule fluency must be exercised* — Sections 1–2 here — and that
  the real paper's arcs make **later parts consume earlier results** (a
  derivation feeds an implementation feeds a checker), which is the
  texture of p13, p14, and p17.

## Going Deeper

Optional forward pointers along the course map — nothing here is needed
for this unit's practice:

- **`C2-linear-models`**: the sum-of-squares score $Q$ and its Section 5
  gradient become the engine for choosing the best coefficients of a
  linear prediction rule.
- **`C3-gradient-descent`**: p13(c)'s "tiny downhill nudges" grow into a
  full procedure — step sizes, stopping rules, and what can go wrong on
  the way down.
- **`C5-neural-networks`**: layers of squashed weighted sums (Section
  4's pattern, tanh included), differentiated by nothing more than this
  session's chain rule applied route by route — where saturation's
  gradient-killing effect becomes a first-class design concern.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. $4(t^3 + 2t)^3 \cdot (3t^2 + 2)$.
2. $e^{-t^2} \cdot (-2t) = -2t\, e^{-t^2}$.
3. Rate $= 3 \cdot (-2) = -6$; the word is *multiply* — along a route,
   rates multiply.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. Chain: $f_u u' + f_v v' = 1 \cdot 3t^2 + (2v)(2) = 3t^2 + 8t$
   (using $v = 2t$).
   Substitute-first: $f = t^3 + 4t^2$, derivative $3t^2 + 8t$. Agree. ✓
2. If $r_k'(t) = 0$, that input receives no nudge when $t$ moves, so its
   route passes nothing along — its term is (partial) $\times\, 0$.
3. Three terms, one per route; each is (how much $f$ cares about that
   input) $\times$ (how fast that input moves with $t$).

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. $\tanh' = 1 - \tanh^2 = 1 - 0.36 = 0.64$.
2. $\tanh^2 x < 1$ for every real $x$ (since $|N| < D$), so
   $1 - \tanh^2 x > 0$.
3. Largest at $x = 0$, value exactly $1$; as $|x|$ grows,
   $\tanh^2 \to 1$ and the derivative decays to $0$ (saturation).

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. Inner scalar $s = 4w_0 - w_1 + 2w_2$; route through $s$:
   $\partial s/\partial w_1 = -1$, so
   $\partial g/\partial w_1 = \big(1 - \tanh^2(s)\big)(-1)
   = -(1 - \tanh^2 s)$.
2. $1 - \tanh^2(5) \approx 1.8 \times 10^{-4}$ — essentially zero: with
   the squashing saturated, *no* coefficient can move the output more
   than a sliver, because every partial shares that same tiny factor.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Residuals at $w = 1$: $(3 - 2,\; 4 - 1) = (1, 3)$.
   $Q'(1) = -2\,(1 \cdot 2 + 3 \cdot 1) = -10$.
2. $r_n = s_n - w_0 - w_1 h_n$ and
   $\partial r_n/\partial w_0 = -1$ (the $w_0$ route enters behind the
   minus, with coefficient 1), so
   $\partial Q/\partial w_0 = \frac1N \sum_n 2 r_n \cdot (-1)
   = -\frac{2}{N}\sum_n r_n$.
3. $n$ is summed away, $j$ stays free; one free index ranging over the
   $d$ columns gives exactly the $d$ entries of the gradient.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. Residuals at $w = 2$: $(1 - 4,\; 3 - 2) = (-3, 1)$.
   $Q'(2) = -2\,\big((-3)(2) + (1)(1)\big) = -2(-5) = 10$.
   Decode: $s = +1$, $m = 10$, so $s + m = 11$ (option E in Section 6's
   listing).
2. A nonzero integer has exactly one $s \cdot m$ decomposition with
   $m > 0$, so all correct solvers decode identically; $0$ would break it
   because $0 = (+1)\cdot 0 = (-1)\cdot 0$ has no unique sign — so the
   problem's data must keep the answer away from zero.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. `np.dot` is banned by name.
   Legal rewrite: `(r[:, None] * X).sum(axis=0)`.
2. `(40, 1)`, then `(40, 2)` (broadcast), then `(2,)`.
3. The ban is part of the graded function's contract (it is testing
   broadcasting fluency); the checker is scaffolding, and the statement
   explicitly grants it a loop — the exam prints which register applies
   to which part.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. Pitfall 1, the dropped minus from the inner derivative $-X_{n,j}$;
   the fix is one `-` in front of the scale factor.
2. The $2/N$: with $N = 40$, $\;N/2 = 20$ — exactly the observed ratio.
   (Dropping only the 2 would give a ratio of 2; only the $1/N$, a ratio
   of $1/40$.)
3. Sum `axis=0` (down the rows, over $n$) to get shape `(2,)`.
   Summing `axis=1` instead computes $\sum_j r_n X_{n,j}$ — a per-row
   number, $r_n$ times row $n$'s plain (unweighted) sum — shape `(40,)`, which
   violates the contract.

</details>